# HW2 Q2 — Tiled Matrix Multiplication
Open this notebook with its working directory set to the folder containing `matMul.cu`. Run the benchmark cells on a CUDA GPU after reviewing the code. Save screenshots of the two labeled required benchmark outputs. Keep the device and CUDA version with the results.

In [ ]:
import csv
import io
from pathlib import Path
import subprocess

def run(*args):
    print("$ " + " ".join(args), flush=True)
    with subprocess.Popen(
        args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, args)

source = Path("matMul.cu")
if not source.is_file():
    raise FileNotFoundError(
        f"Cannot find {source.name} in {Path.cwd()}. "
        "Upload matMul.cu to the notebook runtime or change its working directory."
    )

run("nvidia-smi")
run("nvcc", "--version")
# Use the GPU architecture reported by nvidia-smi.
capability = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    text=True,
).splitlines()[0].strip()
arch = "sm_" + capability.replace(".", "")
command = ["nvcc", "-O3", "-lineinfo", f"-arch={arch}", str(source), "-o", "matMul"]
run(*command)

: 

: 

## Boundary check

In [ ]:
run("./matMul", "17", "19", "23")

## Required run 1 — 1024 × 1024 by 1024 × 1024

In [ ]:
run("./matMul", "1024", "1024", "1024")

## Required run 2 — 513 × 8192 by 8192 × 1023

In [ ]:
run("./matMul", "513", "8192", "1023")

## Nsight Compute
In the two reports, read Achieved Occupancy from Occupancy and per-block shared memory from Launch Statistics. Match the four launches by kernel name and grid/block configuration.

In [ ]:
run("ncu", "--version")
results = Path("q2_results")
results.mkdir(exist_ok=True)
for m, k, n in [(1024, 1024, 1024), (513, 8192, 1023)]:
    report = results / f"ncu_{m}_{k}_{n}"
    run("ncu", "--section", "LaunchStats", "--section", "Occupancy",
        "--clock-control", "none", "--force-overwrite", "-o", str(report),
        "./matMul", str(m), str(k), str(n), "--profile")
    run("ncu", "--import", str(report) + ".ncu-rep", "--page", "details")

## Nine-size runtime sweep

In [ ]:
results = Path("q2_results")
results.mkdir(exist_ok=True)
sizes = [32, 64, 128, 192, 256, 384, 512, 768, 1024]
columns = ["M", "K", "N", "cpu_ms", "gemm_ms", "tiled_8x8_ms",
           "tiled_16x16_ms", "tiled_32x8_ms"]
rows = []
for size in sizes:
    output = subprocess.check_output(
        ["./matMul", str(size), str(size), str(size), "--csv"], text=True,
    )
    rows.append(next(csv.reader(io.StringIO(output))))
with (results / "timings.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(columns)
    writer.writerows(rows)

In [ ]:
import matplotlib.pyplot as plt

with (Path("q2_results") / "timings.csv").open() as stream:
    data = list(csv.DictReader(stream))

fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
bar_width = 0.16
for index, (column, label) in enumerate([
    ("cpu_ms", "CPU"), ("gemm_ms", "gemm (16×16)"),
    ("tiled_8x8_ms", "Tiled 8×8"), ("tiled_16x16_ms", "Tiled 16×16"),
    ("tiled_32x8_ms", "Tiled 32×8"),
]):
    positions = [i + (index - 2) * bar_width for i in range(len(data))]
    ax.bar(positions, [float(row[column]) for row in data], bar_width, label=label)
ax.set_yscale("log")
ax.set_xticks(range(len(data)), [row["M"] for row in data])
ax.set_xlabel("Square matrix side length")
ax.set_ylabel("Runtime (ms, log scale)")
ax.legend()
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
fig.savefig(Path("q2_results") / "runtime_comparison.png", dpi=220)
plt.show()

## Terminal commands

Run these Bash cells in order from the directory containing `matMul.cu`. This example uses `sm_75` for the Tesla T4. The executable notebook cells above select the architecture from the GPU.

### Compile and run

In [ ]:
%%bash
set -euo pipefail
nvcc -O3 -lineinfo -arch=sm_75 matMul.cu -o matMul
./matMul 17 19 23
./matMul 1024 1024 1024
./matMul 513 8192 1023

### Profile both required inputs

In [ ]:
%%bash
set -euo pipefail
mkdir -p q2_results
ncu --section LaunchStats --section Occupancy --clock-control none --force-overwrite -o q2_results/ncu_1024_1024_1024 ./matMul 1024 1024 1024 --profile
ncu --section LaunchStats --section Occupancy --clock-control none --force-overwrite -o q2_results/ncu_513_8192_1023 ./matMul 513 8192 1023 --profile
ncu --import q2_results/ncu_1024_1024_1024.ncu-rep --page details
ncu --import q2_results/ncu_513_8192_1023.ncu-rep --page details

### Export the nine-size runtime sweep

In [ ]:
%%bash
set -euo pipefail
mkdir -p q2_results
(
    set -e
    printf '%s\n' 'M,K,N,cpu_ms,gemm_ms,tiled_8x8_ms,tiled_16x16_ms,tiled_32x8_ms'
    for size in 32 64 128 192 256 384 512 768 1024; do
        ./matMul "$size" "$size" "$size" --csv
    done
) > q2_results/timings.csv

Run the plot cell above after exporting the CSV to generate `q2_results/runtime_comparison.png`.